**ROI counters and mosaic workflow over a set of Laue images (2D map)**

Same core workflow (`LaueTools.scripts.workflows.ROICountersWorkflow`) as in:
- **GUI**: PeakSearchGUI, tab *Mosaic & Monitor* (*Computing / Run on*)
- **command line**: `python -m LaueTools.scripts.workflowlauncher submit config.json --resource magnifix --wait`
- **this notebook**

Images are processed in parallel, in this kernel, in a background process on this machine, or in a SLURM job on an ESRF partition.

Counters (in each ROI of each image): `mosaic` (ROI pixel intensities), `mean`, `max`, `ptp`, `sum`, `pixelval` (ROI center pixel), `XYmax` (position of max), `XYcentroid` (center of mass above ROI minimum), `fit` (2D gaussian fit, see `FIT_COLUMNS`).

In [ ]:
# %matplotlib widget   # for jupyterlab interactive plots
import os
import numpy as np
import matplotlib.pyplot as plt

import LaueTools.logfile_reader as iohdf5
import LaueTools.scripts.workflows as wf
import LaueTools.scripts.workflowlauncher as WL

# 0 - Parameters

(cell tagged `parameters`)

In [ ]:
# experiment and scan (BLISS hdf5 logfile of a dataset)
CCDLabel = 'EIGER_4MCdTe'
h5logfile = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<dataset>/<dataset>.h5'
scan_idx = 0  # index of scan in logfile.dfallscans

# ROIs: list of (X, Y) detector pixels, half sizes along X and Y
roicenters = [[1177, 1803]]
boxsize_X, boxsize_Y = 8, 6

# computation: in this kernel (3a) and/or as a job (3b)
RUN_IN_KERNEL = True
resource = 'local'  # or 'magnifix', 'nice-hpc7', ... (see WL.COMPUTING_RESOURCES)
nbcpus = 32

# output folder: 'image' (folder of images, default), 'mirror' (PROCESSED_DATA mirror folder) or path of a folder
OUTPUT_FOLDER_MODE = 'image'

# 1 - Scan parameters from BLISS hdf5 logfile

In [ ]:
logfile = iohdf5.H5file(h5logfile, CCDLabel)
logfile.getscans(verbose=0)
logfile.dfallscans[['fullcommand', 'imagefolder']]

In [ ]:
d = logfile.build_dict_scan(scan_idx, verbose=False)
print(d['fullcommand'], '\nmap dimensions (fast, slow):', d['mapdimensions'], '\nimages folder:', d['folder'])

# 2 - ROIs and counters

`roicenters`: list of (X, Y) detector pixels (one ROI as in the GUI, or many ROIs e.g. from a peak search)

In [ ]:
params = dict(d)
params['roicenters'] = roicenters
params['boxsize_X'], params['boxsize_Y'] = boxsize_X, boxsize_Y
params['counters'] = ['mosaic', 'mean', 'max', 'ptp', 'XYcentroid', 'fit']
params['fitparameters'] = {'peaksizeStart': 2.}

print('possible counters:', list(wf.ROI_COUNTERS))

# 3a - Compute in this kernel (blocking)

In [ ]:
if RUN_IN_KERNEL:
    roiworkflow = wf.ROICountersWorkflow(params)
    results = roiworkflow.run_roi_counters_workflow(nb_cpus=nbcpus, progressbar=True)

# 3b - OR launch a job on this machine or on a SLURM partition (as the GUI does)

Results (.h5), configuration, log and status files are written in the output folder: folder of images (default), mirror folder in PROCESSED_DATA, or any folder (`OUTPUT_FOLDER_MODE`, same choice as in the GUI). If the folder is not writable, the PROCESSED_DATA mirror folder is used.
For SLURM, `ssh {submit host}` must work without password if SLURM is not available on this machine.

In [ ]:
print('computing resources:', list(WL.COMPUTING_RESOURCES))

outputfolder = WL.output_folder(params['folder'], OUTPUT_FOLDER_MODE)
output = os.path.join(outputfolder, 'roicounters_X%d_Y%d.h5' % tuple(params['roicenters'][0]))

config = WL.make_config(params, output)
job = WL.launch(config, resource=resource, nbcpus=nbcpus)

In [ ]:
job.wait()
results = job.results()

In [ ]:
# equivalent command line (job configuration file already written)
print(f"python -m LaueTools.scripts.workflowlauncher submit {job.files['config']} --resource magnifix --wait")

# 4 - Plots

In [ ]:
mapdims = results['params']['mapdimensions']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, values) in zip(axes, (('max', results['max'][:, 0, 0]),
                                      ('centroid X', results['XYcentroid'][:, 0, 0]),
                                      ('fitted X', results['fit'][:, 0, wf.FIT_COLUMNS.index('X')]))):
    im = ax.imshow(wf.counter_map2D(values, mapdims), origin='lower')
    ax.set_title(title)
    ax.set_xlabel(f"fast axis {d['fastaxis']}")
    ax.set_ylabel(f"slow axis {d['slowaxis']}")
    fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
mosaic = wf.arrange_mosaic2D(results['mosaic'][:, 0], mapdims)['singleimage']
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(mosaic, origin='lower', vmax=np.nanpercentile(mosaic, 99.5))
ax.set_title('mosaic of ROI centered on %s' % results['params']['roicenters'][0])
plt.show()